# Benchmarking



In [1]:
# %load_ext autoreload
# %autoreload 2
from __future__ import annotations

from pprint import pprint
from time import perf_counter

import pandas as pd

from causalchange.config.benchmark_config import BenchmarkConfig, SpaceTimeAlgoConfig
from experiments.benchmarks.run_methods import (
    iter_valid_configs,
    run_algo,
    run_sampling,
    run_scoring,
)


def supported_score_type(score_type: str) -> bool:
    """Return whether the current benchmark config accepts this score_type."""
    try:
        SpaceTimeAlgoConfig.model_validate(
            {
                "name": "spacetime",
                "score_type": score_type,
            }
        )
        return True
    except Exception:
        return False


def print_metric_group(metrics: dict[str, float], group: str) -> None:
    prefixes_by_group = {
        "summary": ("summary_",),
        "wcg": ("wcg_",),
        "changepoint": ("changepoint_",),
        "partition": ("context_partition_", "regime_partition_"),
        "all": ("summary_", "wcg_", "changepoint_", "context_partition_", "regime_partition_"),
    }
    prefixes = prefixes_by_group[group]
    for key in sorted(metrics):
        if key in {"edge_f1", "skel_f1", "shd", "time_s"} or key.startswith(prefixes):
            print(f"{key:34s} {metrics[key]}")


def selected_metrics(metrics: dict[str, float]) -> dict[str, float]:
    """Compact row for tables."""
    keys = [
        "summary_edge_f1",
        "summary_skel_f1",
        "summary_shd",
        "wcg_edge_f1",
        "wcg_skel_f1",
        "wcg_shd",
        "changepoint_f1",
        "changepoint_n_true",
        "changepoint_n_est",
        "context_partition_ari",
        "regime_partition_ari",
        "time_s",
    ]
    return {key: metrics.get(key, float("nan")) for key in keys}


def run_config_once(cfg: BenchmarkConfig):
    """Fit once and return sample, estimator, metrics, and graph."""
    sample = run_sampling(cfg.data)
    t0 = perf_counter()
    est = run_algo(sample, cfg.data, cfg.algo)
    t1 = perf_counter()
    metrics, est_graph = run_scoring(sample, est, return_nx=True)
    metrics["time_s"] = float(t1 - t0)
    return sample, est, metrics, est_graph


def run_grid_to_frame(grid: dict, *, max_configs: int | None = None, verbose: bool = True) -> pd.DataFrame:
    rows = []
    for idx, cfg in enumerate(iter_valid_configs(grid)):
        if max_configs is not None and idx >= max_configs:
            break

        if verbose:
            print(
                f"\n[{idx + 1}] setting={cfg.data.setting}, score={cfg.algo.score_type}, "
                f"mode={cfg.algo.changepoint_mode}"
            )

        sample, est, metrics, est_graph = run_config_once(cfg)

        row = {
            "setting": cfg.data.setting,
            "score_type": cfg.algo.score_type,
            "changepoint_mode": cfg.algo.changepoint_mode,
            "n_nodes": cfg.data.n_nodes,
            "edge_prob": cfg.data.edge_prob,
            "tau_max": cfg.data.tau_max,
            "nonlinearity": cfg.data.nonlinearity,
            "n_changepoints": cfg.data.n_changepoints,
        }
        if cfg.data.setting == "time":
            row["n_samples"] = cfg.data.n_samples
        else:
            row["n_contexts"] = cfg.data.n_contexts
            row["n_samples_per_context"] = cfg.data.n_samples_per_context
            row["n_context_clusters"] = cfg.data.n_context_clusters

        row.update(selected_metrics(metrics))
        rows.append(row)

        if verbose:
            print_metric_group(metrics, "all")
            print("true changepoints:", sample.spacetime.changepoints)
            print("est changepoints:", est.changepoints_)
            print("est WCG edges:")
            pprint(sorted(est_graph.edges()))

    return pd.DataFrame(rows)

## Presets

In [4]:
HARDNESS_PRESETS = {
    "easy_linear": {
        "n_nodes": 3,
        "edge_prob": 0.25,
        "tau_max": 1,
        "n_changepoints": 1,
        "n_regimes": 2,
        "min_segment_length": 150,
        "nonlinearity": "lin",
        "weight_scale": 0.25,
        "noise_scale": 0.40,
        "mechanism_change_fraction": 0.50,
        "mechanism_shift_scale": 1.25,
        "n_samples": 400,
        "n_samples_per_context": 400,
        "n_contexts": 4,
        "n_context_clusters": 2,
    },
    "medium_nonlinear": {
        "n_nodes": 5,
        "edge_prob": 0.30,
        "tau_max": 1,
        "n_changepoints": 2,
        "n_regimes": 2,
        "min_segment_length": 200,
        "nonlinearity": "tanh",
        "weight_scale": 0.25,
        "noise_scale": 0.50,
        "mechanism_change_fraction": 0.50,
        "mechanism_shift_scale": 1.00,
        "n_samples": 800,
        "n_samples_per_context": 800,
        "n_contexts": 4,
        "n_context_clusters": 2,
    },
    "hard_nonlinear": {
        "n_nodes": 5,
        "edge_prob": 0.40,
        "tau_max": 2,
        "n_changepoints": 2,
        "n_regimes": 2,
        "min_segment_length": 300,
        "nonlinearity": "tanh",
        "weight_scale": 0.20,
        "noise_scale": 0.70,
        "mechanism_change_fraction": 0.50,
        "mechanism_shift_scale": 0.75,
        "n_samples": 1200,
        "n_samples_per_context": 1200,
        "n_contexts": 5,
        "n_context_clusters": 2,
    },
}

SCORE_SETUPS = {
    "linear": "lin",
    "rff": "ff",
    "gp": "gp",
}

## oracle vs full



In [7]:
def spacetime_grid(
    *,
    preset_name: str,
    score_type: str,
    mode: str,
    settings: list[str] | None = None,
    seeds: list[int] | None = None,
) -> dict:
    if settings is None:
        settings = ["time", "time-contexts"]
    if seeds is None:
        seeds = [1]

    preset = HARDNESS_PRESETS[preset_name]

    if mode == "oracle_graph":
        algo_opts = {
            "changepoint_mode": ["oracle"],
        }
    elif mode == "changepoint":
        algo_opts = {
            "changepoint_mode": ["detect"],
        }
    elif mode == "full":
        algo_opts = {
            "changepoint_mode": ["detect"],
        }
    else:
        raise ValueError(f"Unknown mode: {mode!r}")

    return {
        "data": {
            "setting": settings,
            "seed": seeds,
            "n_nodes": [preset["n_nodes"]],
            "edge_prob": [preset["edge_prob"]],
            "tau_max": [preset["tau_max"]],
            "n_changepoints": [preset["n_changepoints"]],
            "n_regimes": [preset["n_regimes"]],
            "min_segment_length": [preset["min_segment_length"]],
            "nonlinearity": [preset["nonlinearity"]],
            "weight_scale": [preset["weight_scale"]],
            "noise_scale": [preset["noise_scale"]],
            "mechanism_change_fraction": [preset["mechanism_change_fraction"]],
            "mechanism_shift_scale": [preset["mechanism_shift_scale"]],
            "n_samples": [preset["n_samples"]],
            "n_contexts": [preset["n_contexts"]],
            "n_samples_per_context": [preset["n_samples_per_context"]],
            "n_context_clusters": [preset["n_context_clusters"]],
            "context_col": ["context"],
        },
        "algo": {
            "name": ["spacetime"],
            "score_type": [score_type],
            **algo_opts,
        },
    }


def run_suite(
    *,
    preset_name: str,
    score_name: str,
    modes: list[str] | None = None,
    settings: list[str] | None = None,
    seeds: list[int] | None = None,
    verbose: bool = False,
) -> pd.DataFrame:
    score_type = SCORE_SETUPS[score_name]
    if not supported_score_type(score_type):
        print(f"Skipping unsupported score setup {score_name!r} / score_type={score_type!r}.")
        return pd.DataFrame()

    if modes is None:
        modes = ["oracle_graph", "changepoint", "full"]

    frames = []
    for mode in modes:
        print(f"\n### preset={preset_name}, score={score_name}, mode={mode}")
        grid = spacetime_grid(
            preset_name=preset_name,
            score_type=score_type,
            mode=mode,
            settings=settings,
            seeds=seeds,
        )
        frame = run_grid_to_frame(grid, verbose=verbose)
        frame.insert(0, "preset", preset_name)
        frame.insert(1, "score_name", score_name)
        frame.insert(2, "mode", mode)
        frames.append(frame)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 3. easy x linear



In [8]:
easy_linear_lin = run_suite(
    preset_name="easy_linear",
    score_name="linear",
    settings=["time", "time-contexts"],
    seeds=[1],
    verbose=True,
)

easy_linear_lin


### preset=easy_linear, score=linear, mode=oracle_graph

[1] setting=time, score=lin, mode=oracle, ctx=False, reg=False
changepoint_f1                     1.0
changepoint_mean_abs_error         0.0
changepoint_n_est                  1.0
changepoint_n_true                 1.0
changepoint_precision              1.0
changepoint_recall                 1.0
edge_f1                            0.5
shd                                2.0
skel_f1                            0.6666666666666666
summary_edge_f1                    0.5
summary_edge_precision             0.5
summary_edge_recall                0.5
summary_shd                        2.0
summary_skel_f1                    0.6666666666666666
summary_skel_precision             0.5
summary_skel_recall                1.0
time_s                             0.21219830000336515
wcg_edge_f1                        0.0
wcg_edge_precision                 0.0
wcg_edge_recall                    0.0
wcg_shd                            3.0
wcg_skel_f1   

,preset,score_name,mode,setting,score_type,changepoint_mode,detect_contexts,detect_regimes,n_nodes,edge_prob,...,wcg_shd,changepoint_f1,changepoint_n_true,changepoint_n_est,context_partition_ari,regime_partition_ari,time_s,n_contexts,n_samples_per_context,n_context_clusters
0,easy_linear,linear,oracle_graph,time,lin,oracle,False,False,3,0.25,...,3.0,1.0,1.0,1.0,NaN,NaN,0.212198,NaN,NaN,NaN
1,easy_linear,linear,oracle_graph,time-contexts,lin,oracle,False,False,3,0.25,...,3.0,1.0,1.0,1.0,NaN,NaN,0.635520,4.0,400.0,2.0
2,easy_linear,linear,changepoint,time,lin,detect,False,False,3,0.25,...,1.0,0.0,1.0,1.0,NaN,NaN,5.060974,NaN,NaN,NaN
3,easy_linear,linear,changepoint,time-contexts,lin,detect,False,False,3,0.25,...,3.0,1.0,1.0,1.0,NaN,NaN,7.569581,4.0,400.0,2.0
4,easy_linear,linear,full,time,lin,detect,True,True,3,0.25,...,2.0,0.0,1.0,1.0,1.000000,0.333333,5.867368,NaN,NaN,NaN
5,easy_linear,linear,full,time-contexts,lin,detect,True,True,3,0.25,...,0.0,0.0,1.0,1.0,0.666667,0.907270,25.739759,4.0,400.0,2.0


## 4. easy x RFF

In [ ]:
easy_linear_rff = run_suite(
    preset_name="easy_linear",
    score_name="rff",
    settings=["time", "time-contexts"],
    seeds=[1],
    verbose=False,
)

print(easy_linear_rff)

In [ ]:
easy_rff_many = run_suite(
    preset_name="easy_linear",
    score_name="rff",
    settings=["time", "time-contexts"],
    seeds=[1, 2, 3, 4, 5],
    verbose=False,
)

easy_rff_many.groupby(["mode", "setting"])[
    [
        "summary_edge_f1",
        "wcg_edge_f1",
        "wcg_shd",
        "changepoint_f1",
        "context_partition_ari",
        "regime_partition_ari",
        "time_s",
    ]
].mean()


### preset=easy_linear, score=rff, mode=oracle_graph


In [ ]:
for cfg in iter_valid_configs(
    spacetime_grid(
        preset_name="easy_linear",
        score_type="ff",
        mode="changepoint",
        settings=["time", "time-contexts"],
        seeds=[1, 2, 3, 4, 5],
    )
):
    sample, est, metrics, _ = run_config_once(cfg)
    print(
        cfg.data.setting,
        "seed=",
        cfg.data.seed,
        "true=",
        sample.spacetime.changepoints,
        "est=",
        est.changepoints_,
        "f1=",
        metrics["changepoint_f1"],
    )

## 5. Medium x RFF




In [12]:
medium_rff = run_suite(
    preset_name="medium_nonlinear",
    score_name="rff",
    settings=["time", "time-contexts"],
    seeds=[1],
    verbose=False,
)

print(medium_rff)


### preset=medium_nonlinear, score=rff, mode=oracle_graph


KeyboardInterrupt: 

## 6. Medium x linear



In [ ]:
medium_linear_score = run_suite(
    preset_name="medium_nonlinear",
    score_name="linear",
    settings=["time", "time-contexts"],
    seeds=[1],
    verbose=False,
)

medium_linear_score

## 7. easy x gp


In [ ]:
mdl_results = run_suite(
    preset_name="easy_linear",
    score_name="gp",
    settings=["time", "time-contexts"],
    seeds=[1],
    verbose=False,
)

mdl_results

## 8. Hard x RFF


In [ ]:
# hard_rff = run_suite(
#     preset_name="hard_nonlinear",
#     score_name="rff",
#     settings=["time-contexts"],
#     seeds=[1],
#     verbose=False,
# )
# hard_rff


## comparison



In [ ]:
frames = [
    name
    for name in [
        globals().get("easy_linear_lin"),
        globals().get("easy_linear_rff"),
        globals().get("medium_rff"),
        globals().get("medium_linear_score"),
        globals().get("mdl_results"),
        globals().get("hard_rff"),
    ]
    if isinstance(name, pd.DataFrame) and not name.empty
]

comparison = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if not comparison.empty:
    display_cols = [
        "preset",
        "score_name",
        "mode",
        "setting",
        "summary_edge_f1",
        "wcg_edge_f1",
        "changepoint_f1",
        "context_partition_ari",
        "regime_partition_ari",
        "time_s",
    ]
    display(comparison[[col for col in display_cols if col in comparison.columns]])
else:
    print("No result frames available yet.")